In [10]:
import pandas as pd
import re
import json
import time

In [ ]:
df = pd.read_csv("../tables/resources-inventory-Mar_03_2026.tab", sep='\t')

In [12]:
df.columns

Index(['Resource Name', 'Abbreviation', 'Coarse Data Modality',
       'Granular Data Modality', 'Diseases Included', 'Sample Size',
       'Access URL', 'FAIR Compliance Notes', 'Date added to catalog',
       'Reviewer', 'Alternative URLs', 'Resource Type', 'Remove', 'Notes',
       'new_corpus'],
      dtype='object')

In [13]:
# Normalize Coarse Data Modality: strip list artifacts, fix typos/case, apply mapping
mapping = {
    'Clinical': 'clinical',
    'Genetics': 'genetics',
    'Genomics': 'genetics',
    'genomics': 'genetics',
    'genetic': 'genetics',
    'Transcriptomics': 'transcriptomics',
    'transcriptomic': 'transcriptomics',
    'trasncriptomics': 'transcriptomics',
    'Proteomics': 'proteomics',
    'proteomcis': 'proteomics',
    'Metabolomics': 'metabolomics',
    'Epigenomics': 'epigenomics',
    'epigenetics': 'epigenomics',
    'Imaging': 'imaging',
    'Other': 'other',
    'cognitive': 'skip',
    'immunological': 'skip',
    'omics': 'skip',
}

def parse_coarse(val):
    if pd.isna(val):
        return ''
    s = str(val).strip()
    # Strip list brackets and internal quotes
    s = s.strip("[]").replace("'", "").replace('"', '')
    terms = [t.strip() for t in s.split(',') if t.strip()]
    normalized = [mapping.get(t, t) for t in terms]
    # Drop skipped terms, deduplicate while preserving order
    seen = set()
    result = []
    for t in normalized:
        if t == 'skip' or t in seen:
            continue
        seen.add(t)
        result.append(t)
    return ', '.join(result)

df['Coarse Data Modality'] = df['Coarse Data Modality'].apply(parse_coarse)

# Verify
from collections import Counter
all_types = []
for val in df['Coarse Data Modality']:
    all_types.extend([t.strip() for t in str(val).split(',') if t.strip()])
for val, count in Counter(all_types).most_common():
    print(f"  {val}: {count}")


  clinical: 183
  genetics: 86
  transcriptomics: 86
  longitudinal: 75
  imaging: 50
  proteomics: 50
  metabolomics: 23
  epigenomics: 21


In [ ]:
# write to tsv as .tab
df.to_csv("../tables/resources-inventory-Mar_11_2026.tab", sep='\t', index=False)